In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori as ap, association_rules as ap_rl
import numpy as np
from icecream import ic
import os
import tensorflow as tf
import keras
from keras import layers
from datetime import date, time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import plotly.graph_objs as go
import networkx as nx
import random  
import string  
import pandas as pd  
from datetime import datetime, timedelta 

In [21]:
export = pd.read_csv("export_nodup.csv")
export = export.drop(['Unnamed: 0','journey_steps_until_end','customer_id'], axis=1)
export.head(20)



,account_id,ed_id,event_name,Date,Time
0,1773350293,12,application_web_approved,2023-03-22,08:45:22
1,1773350293,19,application_web_view,2023-03-22,13:32:10
2,1773350293,3,application_web_submit,2023-03-22,13:32:10
3,1773350293,2,campaign_click,2023-03-22,14:45:22
4,1773350293,19,application_web_view,2023-07-27,14:57:56
5,1773350293,19,application_web_view,2023-08-29,16:01:06
6,383997507,4,browse_products,2021-11-04,14:11:15
7,383997507,4,browse_products,2021-11-04,14:11:29
8,383997507,4,browse_products,2021-11-04,14:12:10
9,383997507,4,browse_products,2021-11-04,14:12:21


In [22]:
# Drop duplicates to keep the original order
result = export.drop_duplicates(subset=['account_id', 'ed_id','Date','Time'])

result["timestamp"] = pd.to_datetime(result['Date'] + ' ' + result['Time'])


/var/folders/t7/frqd8nlj0y31l9jp9ymknvf40000gn/T/ipykernel_17400/3548014825.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result["timestamp"] = pd.to_datetime(result['Date'] + ' ' + result['Time'])


In [23]:
result.head(20)

,account_id,ed_id,event_name,Date,Time,timestamp
0,1773350293,12,application_web_approved,2023-03-22,08:45:22,2023-03-22 08:45:22
1,1773350293,19,application_web_view,2023-03-22,13:32:10,2023-03-22 13:32:10
2,1773350293,3,application_web_submit,2023-03-22,13:32:10,2023-03-22 13:32:10
3,1773350293,2,campaign_click,2023-03-22,14:45:22,2023-03-22 14:45:22
4,1773350293,19,application_web_view,2023-07-27,14:57:56,2023-07-27 14:57:56
5,1773350293,19,application_web_view,2023-08-29,16:01:06,2023-08-29 16:01:06
6,383997507,4,browse_products,2021-11-04,14:11:15,2021-11-04 14:11:15
7,383997507,4,browse_products,2021-11-04,14:11:29,2021-11-04 14:11:29
8,383997507,4,browse_products,2021-11-04,14:12:10,2021-11-04 14:12:10
9,383997507,4,browse_products,2021-11-04,14:12:21,2021-11-04 14:12:21


In [24]:
result = result.drop(['Date','Time'], axis=1)


In [25]:
result

,account_id,ed_id,event_name,timestamp
0,1773350293,12,application_web_approved,2023-03-22 08:45:22
1,1773350293,19,application_web_view,2023-03-22 13:32:10
2,1773350293,3,application_web_submit,2023-03-22 13:32:10
3,1773350293,2,campaign_click,2023-03-22 14:45:22
4,1773350293,19,application_web_view,2023-07-27 14:57:56
...,...,...,...,...
55853905,-983311387,29,account_activitation,2021-05-14 00:00:00
55853906,-983311387,5,view_cart,2021-05-15 09:27:47
55853907,-983311387,24,campaignemail_clicked,2021-05-15 14:27:33
55853908,-983311387,27,account_downpaymentcleared,2021-05-16 00:00:00


In [26]:
result.to_csv("DF_TS.csv")

In [27]:
a = pd.read_csv("DF_TS.csv")

In [28]:
a

,Unnamed: 0,account_id,ed_id,event_name,timestamp
0,0,1773350293,12,application_web_approved,2023-03-22 08:45:22
1,1,1773350293,19,application_web_view,2023-03-22 13:32:10
2,2,1773350293,3,application_web_submit,2023-03-22 13:32:10
3,3,1773350293,2,campaign_click,2023-03-22 14:45:22
4,4,1773350293,19,application_web_view,2023-07-27 14:57:56
...,...,...,...,...,...
55853854,55853905,-983311387,29,account_activitation,2021-05-14 00:00:00
55853855,55853906,-983311387,5,view_cart,2021-05-15 09:27:47
55853856,55853907,-983311387,24,campaignemail_clicked,2021-05-15 14:27:33
55853857,55853908,-983311387,27,account_downpaymentcleared,2021-05-16 00:00:00
